In [1]:
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

sys.path.append("..")

from src.load import loadMatches, loadEvents, loadFrames, LEAKS

# Cleaning the passes
We only use the matches that have 360 data, because the pass model needs to know where the other players were standing.

In [2]:
matches = loadMatches()
matches360 = matches[matches["has360"] == True].copy()

print("All matches:", len(matches))
print("Matches with 360 data:", len(matches360))
print(matches360.groupby(["competition", "season"]).size())

All matches: 3961
Matches with 360 data: 426
competition                                     season   
Africa - African Cup of Nations                 2023          1
Europe - UEFA Euro                              2020         51
                                                2024         51
Europe - UEFA Women's Euro                      2022         31
                                                2025         31
France - Ligue 1                                2021/2022    26
                                                2022/2023    32
Germany - 1. Bundesliga                         2023/2024    34
International - FIFA World Cup                  2022         64
International - Women's World Cup               2023         64
Spain - La Liga                                 2020/2021    35
United States of America - Major League Soccer  2023          6
dtype: int64


## Looking at the passes in one match first
The 2022 World Cup final, so we can see the mess before cleaning everything.

In [3]:
FINAL = 3869685

final = loadEvents(FINAL)
rawPasses = final[final["type"] == "Pass"].copy()

print("Events in the final:", len(final))
print("Passes in the final:", len(rawPasses))

Events in the final: 4407
Passes in the final: 1263


In [4]:
print(rawPasses["pass_outcome"].value_counts(dropna=False))

pass_outcome
NaN             994
Incomplete      232
Out              20
Unknown          10
Pass Offside      7
Name: count, dtype: int64


A blank outcome is a completed pass. StatsBomb only writes an outcome when something went wrong. So dropna() is not an option here.

In [5]:
print("Passes before dropna():", len(rawPasses))
print("Passes after dropna():", len(rawPasses.dropna()))

Passes before dropna(): 1263
Passes after dropna(): 0


In [6]:
print("under_pressure values:")
print(rawPasses["under_pressure"].value_counts(dropna=False))

under_pressure values:
under_pressure
NaN     1148
True     115
Name: count, dtype: int64


under_pressure is only True or blank. Blank means the passer was not under pressure, so it should become 0, not be removed.

## The 360 frames of the final

In [7]:
frames = loadFrames(FINAL)

print("360 rows:", len(frames))
print("Events with a frame:", frames["id"].nunique())
print(frames.head())

360 rows: 61451
Events with a frame: 3683
                                     id  \
0  f651a6c4-55e3-4e0f-a178-59414ba83d6a   
1  f651a6c4-55e3-4e0f-a178-59414ba83d6a   
2  f651a6c4-55e3-4e0f-a178-59414ba83d6a   
3  f651a6c4-55e3-4e0f-a178-59414ba83d6a   
4  f651a6c4-55e3-4e0f-a178-59414ba83d6a   

                                        visible_area  match_id  teammate  \
0  [8.98496759714251, 80.0, 41.4622037211361, 0.0...   3869685      True   
1  [8.98496759714251, 80.0, 41.4622037211361, 0.0...   3869685      True   
2  [8.98496759714251, 80.0, 41.4622037211361, 0.0...   3869685      True   
3  [8.98496759714251, 80.0, 41.4622037211361, 0.0...   3869685      True   
4  [8.98496759714251, 80.0, 41.4622037211361, 0.0...   3869685      True   

   actor  keeper                                  location  
0  False   False    [39.21695730971832, 44.77861301106889]  
1  False   False     [39.2235613627218, 29.49684745718256]  
2  False   False    [42.57667276588051, 67.52374056870202] 

In [8]:
playersPerFrame = frames.groupby("id").size()

print(playersPerFrame.describe())
print("Frames with all 22 players:", (playersPerFrame == 22).sum())

count    3683.000000
mean       16.685039
std         2.984872
min         1.000000
25%        15.000000
50%        17.000000
75%        19.000000
max        21.000000
dtype: float64
Frames with all 22 players: 0


In [9]:
passesWithFrame = rawPasses["id"].isin(frames["id"]).sum()

print("Passes in the final:", len(rawPasses))
print("Passes with a 360 frame:", passesWithFrame)
print("Passes without a 360 frame:", len(rawPasses) - passesWithFrame)

Passes in the final: 1263
Passes with a 360 frame: 998
Passes without a 360 frame: 265


The frames have no player names, only teammate, actor and keeper flags. Players outside the camera are missing, so most frames have fewer than 22 players.

## Cleaning functions

In [10]:
DROPPED = ["Unknown", "Injury Clearance"]

FLAGS = [
    "under_pressure",
    "counterpress",
    "pass_cross",
    "pass_switch",
    "pass_through_ball",
    "pass_aerial_won",
    "pass_inswinging",
    "pass_outswinging",
    "pass_no_touch",
    "pass_miscommunication",
]

COLUMNS = [
    "id",
    "match_id",
    "index",
    "period",
    "minute",
    "second",
    "possession",
    "team",
    "player",
    "player_id",
    "position",
    "play_pattern",
    "x",
    "y",
    "endX",
    "endY",
    "pass_length",
    "pass_angle",
    "pass_height",
    "pass_type",
    "pass_body_part",
    "pass_technique",
] + FLAGS + ["frame", "area", "success"]

In [11]:
def labelPasses(passes):
    """Drop the outcomes we can't trust and make the success label."""
    passes = passes[~passes["pass_outcome"].isin(DROPPED)].copy()
    passes["success"] = passes["pass_outcome"].isna().astype(int)
    return passes


def fillFlags(passes):
    """A missing flag means False, so fill it with 0."""
    for flag in FLAGS:
        if flag in passes.columns:
            passes[flag] = passes[flag].fillna(False).astype(int)
        else:
            passes[flag] = 0
    return passes


def addLocations(passes):
    """Split the start and end points into x and y columns."""
    passes["x"] = passes["location"].apply(lambda spot: spot[0])
    passes["y"] = passes["location"].apply(lambda spot: spot[1])
    passes["endX"] = passes["pass_end_location"].apply(lambda spot: spot[0])
    passes["endY"] = passes["pass_end_location"].apply(lambda spot: spot[1])
    return passes


def groupFrames(frames):
    """Turn the 360 rows of a match into one row per event with a list of players."""
    rows = []
    for eventId, group in frames.groupby("id"):
        players = group[["teammate", "actor", "keeper", "location"]].to_dict("records")
        area = group["visible_area"].iloc[0]
        rows.append({"id": eventId, "frame": players, "area": area})
    return pd.DataFrame(rows)


def addFrames(passes, frames):
    """Attach each pass to its 360 frame and drop passes that have none."""
    grouped = groupFrames(frames)
    passes = passes.merge(grouped, on="id", how="inner")
    return passes


def cleanPasses(events, frames):
    """Run every cleaning step for one match."""
    passes = events[events["type"] == "Pass"].copy()
    passes = passes[passes["period"] != 5].copy()
    passes = labelPasses(passes)
    passes = fillFlags(passes)
    passes = addLocations(passes)
    passes = passes.drop(columns=LEAKS, errors="ignore")
    passes = addFrames(passes, frames)
    for column in COLUMNS:
        if column not in passes.columns:
            passes[column] = None
    passes = passes[COLUMNS].copy()
    return passes

Testing the function on the final.

In [12]:
finalPasses = cleanPasses(final, frames)

print("Clean passes in the final:", len(finalPasses))
print("Completed:", finalPasses["success"].sum())
print("Failed:", (finalPasses["success"] == 0).sum())
print("Leak columns left:", [column for column in LEAKS if column in finalPasses.columns])
print(finalPasses[["x", "y", "endX", "endY", "pass_height", "under_pressure", "success"]].head())

Clean passes in the final: 990
Completed: 797
Failed: 193
Leak columns left: []
      x     y  endX  endY  pass_height  under_pressure  success
0  61.0  40.1  48.0  43.2  Ground Pass               0        1
1  49.7  43.6  96.0   1.8    High Pass               0        0
2  22.7  80.0  36.0  75.2     Low Pass               0        1
3  41.1  73.9  23.2  62.8  Ground Pass               0        1
4  24.7  59.4  21.5  32.2  Ground Pass               0        1


## Counting the mess in every match
While cleaning, we also count each problem so Darpan can put the numbers in the EDA.

In [13]:
def countMess(matchId, events, frames):
    """Count every data problem in one match before cleaning."""
    passes = events[events["type"] == "Pass"]
    frameIds = set(frames["id"])
    playersPerFrame = frames.groupby("id").size()
    actorsPerFrame = frames.groupby("id")["actor"].sum()

    counts = {}
    counts["match_id"] = matchId
    counts["passes"] = len(passes)
    counts["blankOutcome"] = passes["pass_outcome"].isna().sum()
    counts["incomplete"] = (passes["pass_outcome"] == "Incomplete").sum()
    counts["out"] = (passes["pass_outcome"] == "Out").sum()
    counts["offside"] = (passes["pass_outcome"] == "Pass Offside").sum()
    counts["unknown"] = (passes["pass_outcome"] == "Unknown").sum()
    counts["injuryClearance"] = (passes["pass_outcome"] == "Injury Clearance").sum()
    counts["shootout"] = (passes["period"] == 5).sum()
    counts["afterDropna"] = len(passes.dropna())
    counts["pressureBlank"] = passes["under_pressure"].isna().sum() if "under_pressure" in passes.columns else len(passes)
    counts["noFrame"] = (~passes["id"].isin(frameIds)).sum()
    counts["frames"] = len(playersPerFrame)
    counts["fullFrames"] = (playersPerFrame == 22).sum()
    counts["noActor"] = (actorsPerFrame == 0).sum()
    counts["players"] = playersPerFrame.sum()
    return counts

In [14]:
print(countMess(FINAL, final, frames))

{'match_id': 3869685, 'passes': 1263, 'blankOutcome': np.int64(994), 'incomplete': np.int64(232), 'out': np.int64(20), 'offside': np.int64(7), 'unknown': np.int64(10), 'injuryClearance': np.int64(0), 'shootout': np.int64(0), 'afterDropna': 0, 'pressureBlank': np.int64(1148), 'noFrame': np.int64(265), 'frames': 3683, 'fullFrames': np.int64(0), 'noActor': np.int64(0), 'players': np.int64(61451)}


## Cleaning all 426 matches in batches
Saving in batches of 50 matches so a crash doesn't lose everything.

In [15]:
BATCH = 50

batchFolder = Path("../data/clean/pass_batches")
batchFolder.mkdir(parents=True, exist_ok=True)

matchIds = matches360["match_id"].tolist()
messRows = []
failedMatches = []

for start in range(0, len(matchIds), BATCH):
    end = min(start + BATCH, len(matchIds))
    path = batchFolder / f"passes_{start:04d}.parquet"
    batchPasses = []

    for matchId in matchIds[start:end]:
        try:
            events = loadEvents(matchId)
            frames = loadFrames(matchId)
        except (ValueError, KeyError, OSError) as error:
            failedMatches.append((matchId, str(error)))
            continue
        messRows.append(countMess(matchId, events, frames))
        if not path.exists():
            batchPasses.append(cleanPasses(events, frames))

    if path.exists():
        print("Already saved:", start, "to", end)
    else:
        batch = pd.concat(batchPasses, ignore_index=True)
        batch.to_parquet(path, index=False)
        print("Saved matches:", start, "to", end, "| passes:", len(batch))
        del batch
    del batchPasses
    gc.collect()

print("Matches that failed to load:", failedMatches)

Saved matches: 0 to 50 | passes: 43600


Saved matches: 50 to 100 | passes: 44585


Saved matches: 100 to 150 | passes: 50472


Saved matches: 150 to 200 | passes: 42820


Saved matches: 200 to 250 | passes: 45057


Saved matches: 250 to 300 | passes: 45210


Saved matches: 300 to 350 | passes: 38792


Saved matches: 350 to 400 | passes: 39695


Saved matches: 400 to 426 | passes: 21614
Matches that failed to load: [(3845506, "Expecting ',' delimiter: line 92794 column 3 (char 2637824)")]


## Putting the batches together

In [16]:
output = Path("../data/clean/passes.parquet")
batchFiles = sorted(batchFolder.glob("passes_*.parquet"))

writer = None
for path in batchFiles:
    table = pq.read_table(path)
    if writer is None:
        writer = pq.ParquetWriter(output, table.schema)
    writer.write_table(table)
writer.close()

print("Batches:", len(batchFiles))
print("Rows in passes.parquet:", pq.read_metadata(output).num_rows)

Batches: 9
Rows in passes.parquet: 371845


## Checking the saved file

In [17]:
passes = pd.read_parquet(output, columns=["id", "match_id", "success", "under_pressure", "x", "y", "endX", "endY"])

print("Passes:", len(passes))
print("Matches:", passes["match_id"].nunique())
print("Duplicate ids:", passes["id"].duplicated().sum())
print("Missing labels:", passes["success"].isna().sum())
print("Success rate:", round(passes["success"].mean(), 3))
print(passes[["x", "y", "endX", "endY"]].describe().loc[["min", "max"]])

Passes: 371845
Matches: 418
Duplicate ids: 0
Missing labels: 0
Success rate: 0.826


         x     y   endX  endY
min    0.1   0.1    0.1   0.1
max  120.0  80.0  120.0  80.0


In [18]:
schema = pq.ParquetFile(output).schema_arrow

print("Columns:", schema.names)
print("Leak columns left:", [column for column in LEAKS if column in schema.names])
print("Frame type:", schema.field("frame").type)

Columns: ['id', 'match_id', 'index', 'period', 'minute', 'second', 'possession', 'team', 'player', 'player_id', 'position', 'play_pattern', 'x', 'y', 'endX', 'endY', 'pass_length', 'pass_angle', 'pass_height', 'pass_type', 'pass_body_part', 'pass_technique', 'under_pressure', 'counterpress', 'pass_cross', 'pass_switch', 'pass_through_ball', 'pass_aerial_won', 'pass_inswinging', 'pass_outswinging', 'pass_no_touch', 'pass_miscommunication', 'frame', 'area', 'success']
Leak columns left: []
Frame type: list<element: struct<teammate: bool, actor: bool, keeper: bool, location: list<element: double>>>


## Pass mess numbers for the EDA

In [19]:
mess = pd.DataFrame(messRows)
totals = mess.drop(columns=["match_id"]).sum()

findings = {}
findings["Matches with 360 data"] = len(mess)
findings["Raw passes"] = totals["passes"]
findings["Passes left after dropna()"] = totals["afterDropna"]
findings["Blank outcome (completed)"] = totals["blankOutcome"]
findings["Incomplete"] = totals["incomplete"]
findings["Out"] = totals["out"]
findings["Pass Offside"] = totals["offside"]
findings["Unknown (dropped)"] = totals["unknown"]
findings["Injury Clearance (dropped)"] = totals["injuryClearance"]
findings["Shootout passes (dropped)"] = totals["shootout"]
findings["under_pressure blank"] = totals["pressureBlank"]
findings["Passes with no 360 frame (dropped)"] = totals["noFrame"]
findings["360 frames"] = totals["frames"]
findings["Frames with all 22 players"] = totals["fullFrames"]
findings["Frames with no actor"] = totals["noActor"]
findings["Average players per frame"] = round(totals["players"] / totals["frames"], 1)
findings["Clean passes saved"] = len(passes)
findings["Completed share of clean passes"] = round(passes["success"].mean(), 3)

findings = pd.Series(findings, name="value")
print(findings.to_string())

Matches with 360 data                     425.000
Raw passes                             447122.000
Passes left after dropna()                  0.000
Blank outcome (completed)              367768.000
Incomplete                              68464.000
Out                                      7223.000
Pass Offside                             1346.000
Unknown (dropped)                        2014.000
Injury Clearance (dropped)                307.000
Shootout passes (dropped)                   0.000
under_pressure blank                   383905.000
Passes with no 360 frame (dropped)      73482.000
360 frames                            1381467.000
Frames with all 22 players                388.000
Frames with no actor                        2.000
Average players per frame                  15.600
Clean passes saved                     371845.000
Completed share of clean passes             0.826

In [20]:
Path("../outputs").mkdir(exist_ok=True)
findings.to_csv("../outputs/pass_findings.csv")
mess.to_csv("../outputs/pass_mess_by_match.csv", index=False)

print("Saved outputs/pass_findings.csv for Darpan's EDA")

Saved outputs/pass_findings.csv for Darpan's EDA
